# Module 3b: Ray Data - Distributed Datasets

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers Ray Data:
1. Creating and loading datasets
2. Transformations (map, filter, flat_map)
3. Batch operations for ML
4. Integration with pandas and numpy

**Prerequisites**: `pip install "ray[data]"`

## Key Takeaways

- **Ray Data** provides distributed datasets optimized for ML pipelines
- **Lazy evaluation** like Spark - transformations build a plan
- **`map_batches()`** is the key method for efficient ML preprocessing
- **Seamless integration** with pandas, numpy, and Ray Train

In [1]:
!pip install ray "ray[data]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.1 MB/s eta 0:00:00


In [2]:
import ray
import numpy as np
import pandas as pd
import time

# Initialize Ray
if ray.is_initialized():
    ray.shutdown()

ray.init(num_cpus=4, logging_level="WARNING")
print(f"Ray version: {ray.__version__}")

Ray version: 2.55.1


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


---

## 1. Creating Datasets

### From Python Collections

In [3]:
# From a list of dictionaries
data = [
    {"name": "Alice", "age": 25, "city": "NYC"},
    {"name": "Bob", "age": 30, "city": "LA"},
    {"name": "Charlie", "age": 35, "city": "NYC"},
    {"name": "Diana", "age": 28, "city": "Chicago"},
]

ds = ray.data.from_items(data)
print(f"Dataset: {ds}")
print(f"Count: {ds.count()}")
print(f"\nSchema: {ds.schema()}")

2026-05-19 17:52:32,140	INFO logging.py:416 -- Registered dataset logger for dataset dataset_0_0
2026-05-19 17:52:32,187	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-05-19 17:52:32,196	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.
2026-05-19 17:52:34,890	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_0_0 =======
2026-05-19 17:52:34,893	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:52:34,896	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:52:34,897	INFO logging_progress.

Dataset: shape: (4, 3)
╭─────────┬───────┬─────────╮
│ name    ┆ age   ┆ city    │
│ ---     ┆ ---   ┆ ---     │
│ string  ┆ int64 ┆ string  │
╞═════════╪═══════╪═════════╡
│ Alice   ┆ 25    ┆ NYC     │
│ Bob     ┆ 30    ┆ LA      │
│ Charlie ┆ 35    ┆ NYC     │
│ Diana   ┆ 28    ┆ Chicago │
╰─────────┴───────┴─────────╯
(Showing 4 of 4 rows)
Count: 4

Schema: Column  Type
------  ----
name    string
age     int64
city    string


In [4]:
# View first few rows
ds.take(3)

2026-05-19 17:52:35,038	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-05-19 17:52:35,071	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-19 17:52:35,084	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:52:35,088	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=3]
2026-05-19 17:52:35,172	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_1_0 =======
2026-05-19 17:52:35,174	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:52:35,177	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:52:35,182	INFO logging_progress.py:181 -- 
2026-05-19 17:52:35,187	INFO logging_progress.py:231 -- limit=3: 0/1
2

[{'name': 'Alice', 'age': 25, 'city': 'NYC'},
 {'name': 'Bob', 'age': 30, 'city': 'LA'},
 {'name': 'Charlie', 'age': 35, 'city': 'NYC'}]

### From Pandas DataFrame

In [5]:
# Create pandas DataFrame
pdf = pd.DataFrame({
    "temperature": np.random.uniform(20, 35, 1000),
    "humidity": np.random.uniform(30, 90, 1000),
    "pressure": np.random.uniform(990, 1030, 1000),
    "wind_speed": np.random.uniform(0, 30, 1000),
})

# Convert to Ray Dataset
ds_weather = ray.data.from_pandas(pdf)
print(f"Dataset: {ds_weather}")
print(f"\nFirst 3 rows:")
ds_weather.take(3)

2026-05-19 17:52:42,766	INFO logging.py:416 -- Registered dataset logger for dataset dataset_2_0
2026-05-19 17:52:42,782	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_2_0 =======
2026-05-19 17:52:42,783	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:52:42,784	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:52:42,785	INFO logging_progress.py:192 -- ============================================
2026-05-19 17:52:42,792	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_2_0 execution finished in 0.00 seconds
2026-05-19 17:52:42,837	INFO logging.py:416 -- Registered dataset logger for dataset dataset_3_0
2026-05-19 17:52:42,840	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:52:42,841	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_3_0: InputDataBuff

Dataset: shape: (1000, 4)
╭────────────────────┬────────────────────┬────────────────────┬────────────────────╮
│ temperature        ┆ humidity           ┆ pressure           ┆ wind_speed         │
│ ---                ┆ ---                ┆ ---                ┆ ---                │
│ double             ┆ double             ┆ double             ┆ double             │
╞════════════════════╪════════════════════╪════════════════════╪════════════════════╡
│ 30.9521555515635   ┆ 78.43468137094428  ┆ 1021.9990923223387 ┆ 0.7347178074729532 │
│ 27.440294080070913 ┆ 62.8750936208279   ┆ 992.1060677030458  ┆ 15.720995340086885 │
│ 25.766581228814566 ┆ 61.009051901171745 ┆ 999.6081042003347  ┆ 11.502589807111134 │
│ 32.16470494722605  ┆ 47.74293654403187  ┆ 997.4278915364209  ┆ 24.60146636035446  │
│ 30.42215508256979  ┆ 73.08042429568862  ┆ 1009.1156441979489 ┆ 12.327580725555002 │
│ …                  ┆ …                  ┆ …                  ┆ …                  │
│ 25.39646721919503  ┆ 59.57

[{'temperature': 30.9521555515635,
  'humidity': 78.43468137094428,
  'pressure': 1021.9990923223387,
  'wind_speed': 0.7347178074729532},
 {'temperature': 27.440294080070913,
  'humidity': 62.8750936208279,
  'pressure': 992.1060677030458,
  'wind_speed': 15.720995340086885},
 {'temperature': 25.766581228814566,
  'humidity': 61.009051901171745,
  'pressure': 999.6081042003347,
  'wind_speed': 11.502589807111134}]

### From NumPy Arrays

In [6]:
# Create numpy array
np_data = np.random.rand(1000, 5)

# Convert to Ray Dataset
ds_numpy = ray.data.from_numpy(np_data)
print(f"Dataset: {ds_numpy}")
print(f"\nFirst 3 rows:")
ds_numpy.take(3)

2026-05-19 17:52:43,199	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-05-19 17:52:43,221	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-05-19 17:52:43,222	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:52:43,224	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:52:43,224	INFO logging_progress.py:192 -- ============================================
2026-05-19 17:52:43,240	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_4_0 execution finished in 0.00 seconds
2026-05-19 17:52:43,262	INFO logging.py:416 -- Registered dataset logger for dataset dataset_5_0
2026-05-19 17:52:43,265	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_5_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:52:43,266	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_5_0: InputDataBuff

Dataset: shape: (1000, 1)
╭──────────────────────────────────────────╮
│ data                                     │
│ ---                                      │
│ ArrowTensorTypeV2(shape=(5,), dtype=dou… │
╞══════════════════════════════════════════╡
│ [0.89827281 0.49239817 0.79220292 0.989… │
│ [0.14168385 0.50663601 0.80069238 0.523… │
│ [0.93660175 0.71923696 0.48366004 0.805… │
│ [0.78005316 0.22450086 0.7617747  0.444… │
│ [0.38470526 0.93639117 0.0717287  0.355… │
│ …                                        │
│ [0.42227989 0.42070656 0.31165175 0.751… │
│ [0.76874721 0.01484085 0.32056469 0.222… │
│ [0.13162712 0.97145651 0.32869517 0.464… │
│ [0.96485575 0.9677663  0.69484654 0.183… │
│ [0.51209133 0.77187482 0.82109388 0.573… │
╰──────────────────────────────────────────╯
(Showing 10 of 1000 rows)

First 3 rows:


[{'data': array([0.89827281, 0.49239817, 0.79220292, 0.9897519 , 0.38910709])},
 {'data': array([0.14168385, 0.50663601, 0.80069238, 0.52332517, 0.27535646])},
 {'data': array([0.93660175, 0.71923696, 0.48366004, 0.80509081, 0.43127063])}]

### From Files (Parquet, CSV, JSON)

In [7]:
# Save sample data for demonstration
import os
import tempfile

temp_dir = tempfile.mkdtemp()

# Create sample parquet files
for i in range(3):
    chunk = pd.DataFrame({
        "id": range(i*100, (i+1)*100),
        "value": np.random.randn(100),
        "category": np.random.choice(["A", "B", "C"], 100)
    })
    chunk.to_parquet(f"{temp_dir}/data_{i}.parquet")

print(f"Created files in {temp_dir}")
print(os.listdir(temp_dir))

Created files in /tmp/tmpkkfw4tqq
['data_1.parquet', 'data_2.parquet', 'data_0.parquet']


In [9]:
# Read parquet files from the directory
ds_parquet = ray.data.read_parquet(temp_dir)
print(f"Dataset: {ds_parquet}")
print(f"\nFirst 3 rows:")
ds_parquet.take(3)


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:55:48,289	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 0.547.
2026-05-19 17:55:48,291	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 6100806 rows
2026-05-19 17:55:48,306	INFO logging.py:416 -- Registered dataset logger for dataset dataset_7_0
2026-05-19 17:55:48,316	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_7_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:55:48,317	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=3]
2026-05-19 17:55:48,358	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_7_0 =======
2026-05-19 17:55:48,361	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:55:48,362	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:55:48,365	INFO l

Dataset: shape: (?, 3)
╭───────┬────────┬──────────╮
│ id    ┆ value  ┆ category │
│ ---   ┆ ---    ┆ ---      │
│ int64 ┆ double ┆ string   │
╰───────┴────────┴──────────╯
(Dataset isn't materialized)

First 3 rows:


2026-05-19 17:55:55,338	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_7_0 execution finished in 7.02 seconds


[{'id': 100, 'value': -0.28999521425060865, 'category': 'A'},
 {'id': 101, 'value': -0.04405331579188183, 'category': 'C'},
 {'id': 102, 'value': 1.492014024298566, 'category': 'B'}]

---

## 2. Basic Transformations

### map() - Transform Each Row

In [10]:
# Create dataset
ds = ray.data.from_items([{"x": i} for i in range(10)])

# Map: apply function to each row
ds_mapped = ds.map(lambda row: {"x": row["x"], "x_squared": row["x"] ** 2})

print("After map:")
ds_mapped.take(5)

2026-05-19 17:56:16,112	INFO logging.py:416 -- Registered dataset logger for dataset dataset_10_0
2026-05-19 17:56:16,117	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_10_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:16,118	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_10_0: InputDataBuffer[Input] -> LimitOperator[limit=5] -> TaskPoolMapOperator[Map(<lambda>)]
2026-05-19 17:56:16,140	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_10_0 =======
2026-05-19 17:56:16,141	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:16,143	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:16,144	INFO logging_progress.py:181 -- 
2026-05-19 17:56:16,145	INFO logging_progress.py:231 -- limit=5: 0/1
2026-05-19 17:56:16,146	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 9 (0.0B); Resourc

After map:


2026-05-19 17:56:17,909	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_10_0 execution finished in 1.79 seconds


[{'x': 0, 'x_squared': 0},
 {'x': 1, 'x_squared': 1},
 {'x': 4, 'x_squared': 16},
 {'x': 2, 'x_squared': 4},
 {'x': 3, 'x_squared': 9}]

### filter() - Select Rows

In [11]:
# Filter: keep rows matching condition
ds_filtered = ds_mapped.filter(lambda row: row["x"] > 5)

print("After filter (x > 5):")
ds_filtered.take_all()

/usr/local/lib/python3.12/dist-packages/ray/data/dataset.py:1633: UserWarning: Use 'expr' instead of 'fn' when possible for performant filters.
  warnings.warn(
2026-05-19 17:56:17,935	INFO logging.py:416 -- Registered dataset logger for dataset dataset_11_0
2026-05-19 17:56:17,941	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_11_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:17,942	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_11_0: InputDataBuffer[Input] -> TaskPoolMapOperator[Map(<lambda>)->Filter(<lambda>)]
2026-05-19 17:56:17,989	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_11_0 =======
2026-05-19 17:56:17,991	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:17,995	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:17,996	INFO logging_progress.py:181 -- 
2026-05-19 17:56:17,998	INF

After filter (x > 5):


2026-05-19 17:56:18,172	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_11_0 execution finished in 0.23 seconds


[{'x': 6, 'x_squared': 36},
 {'x': 7, 'x_squared': 49},
 {'x': 8, 'x_squared': 64},
 {'x': 9, 'x_squared': 81}]

### flat_map() - One to Many

In [12]:
# Flat map: one row becomes multiple rows
ds_small = ray.data.from_items([{"nums": [1, 2, 3]}, {"nums": [4, 5]}])

ds_flat = ds_small.flat_map(lambda row: [{"num": n} for n in row["nums"]])

print("Original:")
print(ds_small.take_all())
print("\nAfter flat_map:")
print(ds_flat.take_all())

2026-05-19 17:56:18,219	INFO logging.py:416 -- Registered dataset logger for dataset dataset_12_0
2026-05-19 17:56:18,244	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_12_0 =======
2026-05-19 17:56:18,245	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:18,246	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:18,247	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:18,258	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_12_0 execution finished in 0.00 seconds
2026-05-19 17:56:18,285	INFO logging.py:416 -- Registered dataset logger for dataset dataset_13_0
2026-05-19 17:56:18,288	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_13_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:18,290	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_13_0: InputD

Original:
[{'nums': [1, 2, 3]}, {'nums': [4, 5]}]

After flat_map:
[{'num': 1}, {'num': 2}, {'num': 3}, {'num': 4}, {'num': 5}]


---

## 3. Batch Operations (map_batches)

**`map_batches()`** is the most important method for ML preprocessing. It processes multiple rows at once, enabling vectorized operations.

In [13]:
# Create dataset with numeric columns
ds_numeric = ray.data.from_pandas(pd.DataFrame({
    "feature1": np.random.randn(1000),
    "feature2": np.random.randn(1000) * 2 + 5,
    "feature3": np.random.randn(1000) * 0.5 - 1,
}))

print(f"Dataset: {ds_numeric}")

2026-05-19 17:56:18,451	INFO logging.py:416 -- Registered dataset logger for dataset dataset_14_0
2026-05-19 17:56:18,474	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_14_0 =======
2026-05-19 17:56:18,475	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:18,476	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:18,477	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:18,495	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_14_0 execution finished in 0.00 seconds


Dataset: shape: (1000, 3)
╭─────────────────────┬────────────────────┬─────────────────────╮
│ feature1            ┆ feature2           ┆ feature3            │
│ ---                 ┆ ---                ┆ ---                 │
│ double              ┆ double             ┆ double              │
╞═════════════════════╪════════════════════╪═════════════════════╡
│ 0.5733721949530094  ┆ 3.8103681765338706 ┆ -2.0137945736285654 │
│ 0.24426198933251334 ┆ 6.334695826899168  ┆ -0.9353336953978759 │
│ 1.8076278014995446  ┆ 5.046595237540129  ┆ -0.3430538013128118 │
│ 1.3660421937616265  ┆ 4.504038117600551  ┆ -0.9469832257397854 │
│ 0.28901768267201977 ┆ 6.891942141022271  ┆ -1.134130548615272  │
│ …                   ┆ …                  ┆ …                   │
│ 0.34186013711811525 ┆ 2.588865029132673  ┆ -0.7005048612273086 │
│ -0.5442764698066722 ┆ 3.3126842978551183 ┆ -0.5388387465607354 │
│ -0.9915769206799091 ┆ 2.2578128340273573 ┆ -0.6210270415399372 │
│ -0.6236688875769532 ┆ 4.9794194257

In [14]:
# Define batch processing function
def normalize_batch(batch: pd.DataFrame) -> pd.DataFrame:
    """Normalize all columns to [0, 1] range within batch."""
    result = batch.copy()
    for col in batch.columns:
        min_val = batch[col].min()
        max_val = batch[col].max()
        if max_val > min_val:
            result[col] = (batch[col] - min_val) / (max_val - min_val)
    return result

# Apply to batches
ds_normalized = ds_numeric.map_batches(normalize_batch, batch_format="pandas")

print("Normalized sample:")
ds_normalized.take(3)

2026-05-19 17:56:18,547	INFO logging.py:416 -- Registered dataset logger for dataset dataset_16_0
2026-05-19 17:56:18,551	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_16_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:18,553	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_16_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(normalize_batch)] -> LimitOperator[limit=3]
2026-05-19 17:56:18,592	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_16_0 =======
2026-05-19 17:56:18,596	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:18,598	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:18,599	INFO logging_progress.py:181 -- 
2026-05-19 17:56:18,601	INFO logging_progress.py:231 -- MapBatches(normalize_batch): 0/1
2026-05-19 17:56:18,603	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0

Normalized sample:


[{'feature1': 0.5922950145603838,
  'feature2': 0.4183178318711284,
  'feature3': 0.23690408971837315},
 {'feature1': 0.5411425960721475,
  'feature2': 0.6088743123437242,
  'feature3': 0.5650499400133444},
 {'feature1': 0.7841309352620436,
  'feature2': 0.5116381572957,
  'feature3': 0.7452643486954786}]

### Batch Processing with NumPy

In [15]:
# Process as numpy arrays (efficient for numeric operations)
def compute_features_numpy(batch: dict) -> dict:
    """Compute additional features from numpy batch."""
    f1 = batch["feature1"]
    f2 = batch["feature2"]
    f3 = batch["feature3"]

    return {
        "feature1": f1,
        "feature2": f2,
        "feature3": f3,
        "f1_squared": f1 ** 2,
        "f1_f2_product": f1 * f2,
        "feature_sum": f1 + f2 + f3,
    }

ds_features = ds_numeric.map_batches(compute_features_numpy, batch_format="numpy")

print("With additional features:")
ds_features.take(3)

2026-05-19 17:56:18,708	INFO logging.py:416 -- Registered dataset logger for dataset dataset_18_0
2026-05-19 17:56:18,713	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_18_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:18,714	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_18_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(compute_features_numpy)] -> LimitOperator[limit=3]


With additional features:


2026-05-19 17:56:18,761	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_18_0 =======
2026-05-19 17:56:18,767	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:18,770	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:18,771	INFO logging_progress.py:181 -- 
2026-05-19 17:56:18,774	INFO logging_progress.py:231 -- MapBatches(compute_features_numpy): 0/1
2026-05-19 17:56:18,777	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-05-19 17:56:18,778	INFO logging_progress.py:231 -- limit=3: 0/1
2026-05-19 17:56:18,780	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-05-19 17:56:18,783	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:18,837	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_18_0 execution f

[{'feature1': 0.5733721949530094,
  'feature2': 3.8103681765338706,
  'feature3': -2.0137945736285654,
  'f1_squared': 0.3287556739452318,
  'f1_f2_product': 2.1847591649583213,
  'feature_sum': 2.3699457978583145},
 {'feature1': 0.24426198933251334,
  'feature2': 6.334695826899168,
  'feature3': -0.9353336953978759,
  'f1_squared': 0.059663919432676855,
  'f1_f2_product': 1.5473254044947613,
  'feature_sum': 5.643624120833805},
 {'feature1': 1.8076278014995446,
  'feature2': 5.046595237540129,
  'feature3': -0.3430538013128118,
  'f1_squared': 3.267518268754077,
  'f1_f2_product': 9.122365854292735,
  'feature_sum': 6.511169237726862}]

### Controlling Batch Size

In [16]:
def print_batch_info(batch: pd.DataFrame) -> pd.DataFrame:
    """Print info about each batch."""
    print(f"  Processing batch with {len(batch)} rows")
    return batch

# Default batch size
print("Default batch size:")
_ = ds_numeric.map_batches(print_batch_info, batch_format="pandas").take(1)

# Custom batch size
print("\nBatch size = 100:")
_ = ds_numeric.map_batches(
    print_batch_info,
    batch_format="pandas",
    batch_size=100
).take(1)

2026-05-19 17:56:18,879	INFO logging.py:416 -- Registered dataset logger for dataset dataset_20_0
2026-05-19 17:56:18,887	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_20_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:18,888	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_20_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(print_batch_info)] -> LimitOperator[limit=1]


Default batch size:


2026-05-19 17:56:18,929	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_20_0 =======
2026-05-19 17:56:18,930	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:18,933	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:18,936	INFO logging_progress.py:181 -- 
2026-05-19 17:56:18,939	INFO logging_progress.py:231 -- MapBatches(print_batch_info): 0/1
2026-05-19 17:56:18,941	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-05-19 17:56:18,942	INFO logging_progress.py:231 -- limit=1: 0/1
2026-05-19 17:56:18,948	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-05-19 17:56:18,951	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:19,006	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_20_0 execution finishe


Batch size = 100:
(MapBatches(print_batch_info) pid=9991)   Processing batch with 1000 rows


2026-05-19 17:56:19,080	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_22_0 =======
2026-05-19 17:56:19,084	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:19,085	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:19,087	INFO logging_progress.py:181 -- 
2026-05-19 17:56:19,090	INFO logging_progress.py:231 -- MapBatches(print_batch_info): 0/1
2026-05-19 17:56:19,092	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-05-19 17:56:19,095	INFO logging_progress.py:231 -- limit=1: 0/1
2026-05-19 17:56:19,097	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-05-19 17:56:19,100	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:19,172	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_22_0 execution finishe

(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows
(MapBatches(print_batch_info) pid=9991)   Processing batch with 100 rows


---

## 4. Aggregations

In [17]:
# Create dataset with groups
ds_grouped = ray.data.from_items([
    {"city": "NYC", "temperature": 75, "humidity": 60},
    {"city": "NYC", "temperature": 78, "humidity": 55},
    {"city": "LA", "temperature": 85, "humidity": 40},
    {"city": "LA", "temperature": 82, "humidity": 45},
    {"city": "Chicago", "temperature": 68, "humidity": 70},
    {"city": "Chicago", "temperature": 65, "humidity": 75},
])

# Global aggregations
print("Global stats:")
print(f"  Min temperature: {ds_grouped.min('temperature')}")
print(f"  Max temperature: {ds_grouped.max('temperature')}")
print(f"  Mean temperature: {ds_grouped.mean('temperature')}")

Global stats:


2026-05-19 17:56:19,230	INFO logging.py:416 -- Registered dataset logger for dataset dataset_25_0
2026-05-19 17:56:19,240	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=0.0MiB, output=0.0MiB, total=0.0MiB, 
2026-05-19 17:56:19,246	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_25_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:19,248	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_25_0: InputDataBuffer[Input] -> HashAggregateOperator[HashAggregate(key_columns=(), num_partitions=1)] -> LimitOperator[limit=1]
2026-05-19 17:56:19,441	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_25_0 =======
2026-05-19 17:56:19,447	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:19,451	INFO logging_progress.py:227 -- Active & requested resources: 0.01/4 CPU, 0.0B/1.8G

  Min temperature: 65


2026-05-19 17:56:26,756	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_27_0 execution finished in 3.16 seconds
2026-05-19 17:56:26,779	INFO logging.py:416 -- Registered dataset logger for dataset dataset_29_0
2026-05-19 17:56:26,785	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.0GiB): shuffle=0.0MiB, output=0.0MiB, total=0.0MiB, 
2026-05-19 17:56:26,789	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_29_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:26,790	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_29_0: InputDataBuffer[Input] -> HashAggregateOperator[HashAggregate(key_columns=(), num_partitions=1)] -> LimitOperator[limit=1]
2026-05-19 17:56:26,830	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_29_0 =======
2026-05-19 17:56:26,832	INFO logging_progress.py:225 -- Total Pro

  Max temperature: 85


2026-05-19 17:56:29,976	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_29_0 execution finished in 3.19 seconds


  Mean temperature: 75.5


In [18]:
# Group by aggregations
ds_by_city = ds_grouped.groupby("city").mean(["temperature", "humidity"])

print("Grouped by city:")
ds_by_city.take_all()

2026-05-19 17:56:30,008	INFO logging.py:416 -- Registered dataset logger for dataset dataset_30_0
2026-05-19 17:56:30,014	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=6, aggregators=4, dataset (estimate)=0.0GiB): shuffle=0.0MiB, output=0.0MiB, total=0.0MiB, 
2026-05-19 17:56:30,019	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_30_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:30,023	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_30_0: InputDataBuffer[Input] -> HashAggregateOperator[HashAggregate(key_columns=('city',), num_partitions=6)]
2026-05-19 17:56:30,193	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_30_0 =======


Grouped by city:


2026-05-19 17:56:30,199	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:30,205	INFO logging_progress.py:227 -- Active & requested resources: 0.04/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:30,212	INFO logging_progress.py:181 -- 
2026-05-19 17:56:30,218	INFO logging_progress.py:231 -- HashAggregate(key_columns=('city',), num_partitions=6): 0/1
2026-05-19 17:56:30,220	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 5 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-05-19 17:56:30,225	INFO logging_progress.py:231 --     - Shuffle: 0/?
2026-05-19 17:56:30,227	INFO logging_progress.py:231 --     - Aggregation: 0/1
2026-05-19 17:56:30,233	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:40,285	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_30_0 =======
2026-05-19 17:56:40,286	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:40,290	INFO logging_progress.py:227

[{'city': 'LA', 'mean(temperature)': 83.5, 'mean(humidity)': 42.5},
 {'city': 'Chicago', 'mean(temperature)': 66.5, 'mean(humidity)': 72.5},
 {'city': 'NYC', 'mean(temperature)': 76.5, 'mean(humidity)': 57.5}]

---

## 5. Train/Test Split

In [19]:
# Create a larger dataset
ds_full = ray.data.from_pandas(pd.DataFrame({
    "feature": np.random.randn(1000),
    "label": np.random.randint(0, 2, 1000)
}))

# Split into train/test
train_ds, test_ds = ds_full.train_test_split(test_size=0.2, shuffle=True)

print(f"Full dataset: {ds_full.count()} rows")
print(f"Train dataset: {train_ds.count()} rows")
print(f"Test dataset: {test_ds.count()} rows")

2026-05-19 17:56:42,813	INFO logging.py:416 -- Registered dataset logger for dataset dataset_32_0
2026-05-19 17:56:42,819	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_32_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:56:42,820	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_32_0: InputDataBuffer[Input] -> AllToAllOperator[RandomShuffle]
2026-05-19 17:56:42,839	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_32_0 =======
2026-05-19 17:56:42,841	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:42,842	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:42,843	INFO logging_progress.py:181 -- 
2026-05-19 17:56:42,844	INFO logging_progress.py:231 -- RandomShuffle: 0/1
2026-05-19 17:56:42,846	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 1 (15.8KiB); Resources: 0.0 CPU, 0.0B ob

Full dataset: 1000 rows
Train dataset: 800 rows
Test dataset: 200 rows


---

## 6. Converting Back to Pandas/NumPy

In [20]:
# To pandas
pdf_result = train_ds.to_pandas()
print(f"Pandas DataFrame shape: {pdf_result.shape}")
print(pdf_result.head())

2026-05-19 17:56:43,022	INFO logging.py:416 -- Registered dataset logger for dataset dataset_33_0
2026-05-19 17:56:43,042	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_33_0 =======
2026-05-19 17:56:43,043	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:43,044	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:43,045	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:43,055	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_33_0 execution finished in 0.20 seconds


Pandas DataFrame shape: (800, 2)
    feature  label
0  0.528055      1
1 -1.058450      0
2 -0.576667      0
3 -0.216334      1
4  0.985548      1


In [21]:
# Iterate in batches (memory efficient for large datasets)
print("Iterating in batches:")
for i, batch in enumerate(train_ds.iter_batches(batch_size=200, batch_format="pandas")):
    print(f"  Batch {i}: {len(batch)} rows")
    if i >= 2:
        print("  ...")
        break

2026-05-19 17:56:43,079	INFO logging.py:416 -- Registered dataset logger for dataset dataset_33_1
2026-05-19 17:56:43,096	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_33_1 =======
2026-05-19 17:56:43,097	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:43,099	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:43,101	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:43,118	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_33_1 execution finished in 0.20 seconds


Iterating in batches:
  Batch 0: 200 rows
  Batch 1: 200 rows
  Batch 2: 200 rows
  ...


---

## 7. Exercise: Weather Data Pipeline

Create a data processing pipeline for weather data:

In [22]:
# Generate sample weather data
np.random.seed(42)
n_samples = 5000

weather_data = pd.DataFrame({
    "station_id": np.random.choice(["A", "B", "C", "D"], n_samples),
    "temperature_kelvin": np.random.uniform(260, 310, n_samples),  # Kelvin
    "humidity_pct": np.random.uniform(20, 100, n_samples),
    "pressure_hpa": np.random.uniform(980, 1040, n_samples),
    "wind_speed_ms": np.random.uniform(0, 25, n_samples),
})

ds_weather = ray.data.from_pandas(weather_data)
print(f"Weather dataset: {ds_weather}")

2026-05-19 17:56:43,157	INFO logging.py:416 -- Registered dataset logger for dataset dataset_35_0
2026-05-19 17:56:43,174	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_35_0 =======
2026-05-19 17:56:43,175	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:56:43,177	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:56:43,177	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:56:43,185	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_35_0 execution finished in 0.00 seconds


Weather dataset: shape: (5000, 5)
╭────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────╮
│ station_id ┆ temperature_kelvin ┆ humidity_pct       ┆ pressure_hpa       ┆ wind_speed_ms      │
│ ---        ┆ ---                ┆ ---                ┆ ---                ┆ ---                │
│ object     ┆ double             ┆ double             ┆ double             ┆ double             │
╞════════════╪════════════════════╪════════════════════╪════════════════════╪════════════════════╡
│ C          ┆ 301.37594610531517 ┆ 63.43025058886441  ┆ 1028.4028860826836 ┆ 6.013721205364675  │
│ D          ┆ 298.22638973273587 ┆ 86.63148759097835  ┆ 980.9923669573926  ┆ 6.850617350962146  │
│ A          ┆ 288.67644757277867 ┆ 30.00711278691228  ┆ 991.2084738914299  ┆ 23.018448392326405 │
│ C          ┆ 307.8023571805967  ┆ 30.382637257808902 ┆ 1019.2943181270699 ┆ 1.9184878436314057 │
│ C          ┆ 270.0237257764201  ┆ 64.54546882226461  ┆ 1002.5455608682378

In [23]:
# Exercise: Complete the pipeline

def convert_temperature(batch: pd.DataFrame) -> pd.DataFrame:
    """
    Convert temperature from Kelvin to Celsius.
    Celsius = Kelvin - 273.15
    """
    # Your code here
    pass

def compute_heat_index(batch: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a simplified heat index.
    heat_index = temperature_celsius + (humidity_pct * 0.1)
    """
    # Your code here
    pass

def categorize_wind(batch: pd.DataFrame) -> pd.DataFrame:
    """
    Add wind category:
    - calm: < 5 m/s
    - moderate: 5-15 m/s
    - strong: > 15 m/s
    """
    # Your code here
    pass

# Build pipeline
# ds_processed = ds_weather \
#     .map_batches(convert_temperature, batch_format="pandas") \
#     .map_batches(compute_heat_index, batch_format="pandas") \
#     .map_batches(categorize_wind, batch_format="pandas")

# print("Processed sample:")
# ds_processed.take(5)

In [25]:
# Solution

def convert_temperature_solution(batch: pd.DataFrame) -> pd.DataFrame:
    result = batch.copy()
    result["temperature_celsius"] = result["temperature_kelvin"] - 273.15
    return result

def compute_heat_index_solution(batch: pd.DataFrame) -> pd.DataFrame:
    result = batch.copy()
    result["heat_index"] = result["temperature_celsius"] + (result["humidity_pct"] * 0.1)
    return result

def categorize_wind_solution(batch: pd.DataFrame) -> pd.DataFrame:
    result = batch.copy()
    conditions = [
        result["wind_speed_ms"] < 5,
        (result["wind_speed_ms"] >= 5) & (result["wind_speed_ms"] <= 15),
        result["wind_speed_ms"] > 15
    ]
    choices = ["calm", "moderate", "strong"]
    result["wind_category"] = np.select(conditions, choices, default="unknown")
    return result

# Build pipeline
ds_processed = ds_weather \
    .map_batches(convert_temperature_solution, batch_format="pandas") \
    .map_batches(compute_heat_index_solution, batch_format="pandas") \
    .map_batches(categorize_wind_solution, batch_format="pandas")

print("Processed sample:")
for row in ds_processed.take(5):
    print(f"  Station {row['station_id']}: {row['temperature_celsius']:.1f}C, "
          f"Heat Index: {row['heat_index']:.1f}, Wind: {row['wind_category']}")


2026-05-19 17:57:53,826	INFO logging.py:416 -- Registered dataset logger for dataset dataset_43_0
2026-05-19 17:57:53,849	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_43_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:57:53,853	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_43_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(convert_temperature_solution)->MapBatches(compute_heat_index_solution)->MapBatches(categorize_wind_solution)] -> LimitOperator[limit=5]


Processed sample:


2026-05-19 17:57:54,073	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_43_0 =======
2026-05-19 17:57:54,091	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:57:54,104	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:57:54,114	INFO logging_progress.py:181 -- 
2026-05-19 17:57:54,123	INFO logging_progress.py:231 -- MapBatches(convert_temperature_solution)->...->MapBatches(categorize_wind_solution): 0/1
2026-05-19 17:57:54,127	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resources: 1.0 CPU, 0.0B object store
2026-05-19 17:57:54,132	INFO logging_progress.py:231 -- limit=5: 0/1
2026-05-19 17:57:54,133	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-05-19 17:57:54,136	INFO logging_progress.py:192 -- =============================================
2026-05-19 17:57:55,056	INFO streaming_execut

  Station C: 28.2C, Heat Index: 34.6, Wind: moderate
  Station D: 25.1C, Heat Index: 33.7, Wind: moderate
  Station A: 15.5C, Heat Index: 18.5, Wind: strong
  Station C: 34.7C, Heat Index: 37.7, Wind: calm
  Station C: -3.1C, Heat Index: 3.3, Wind: strong


In [26]:
# Analyze by station
print("\nStatistics by station:")
station_stats = ds_processed.groupby("station_id").mean(["temperature_celsius", "heat_index"])

for row in station_stats.take_all():
    print(f"  Station {row['station_id']}: "
          f"Avg Temp: {row['mean(temperature_celsius)']:.1f}C, "
          f"Avg Heat Index: {row['mean(heat_index)']:.1f}")

2026-05-19 17:58:02,813	INFO logging.py:416 -- Registered dataset logger for dataset dataset_44_0
2026-05-19 17:58:02,822	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_44_0. Full logs are in /tmp/ray/session_2026-05-19_17-52-06_396369_8585/logs/ray-data
2026-05-19 17:58:02,825	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_44_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(convert_temperature_solution)->MapBatches(compute_heat_index_solution)->MapBatches(categorize_wind_solution)] -> HashAggregateOperator[HashAggregate(key_columns=('station_id',), num_partitions=1)]
2026-05-19 17:58:02,881	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_44_0 =======
2026-05-19 17:58:02,884	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:58:02,885	INFO logging_progress.py:227 -- Active & requested resources: 0.25/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:58:02,886	INFO logging_progress.py:181 -- 
2026-05-19


Statistics by station:


2026-05-19 17:58:03,918	INFO logging.py:416 -- Registered dataset logger for dataset dataset_8_2
2026-05-19 17:58:03,922	INFO logging.py:424 -- dataset_8_2 registers for logging while another dataset dataset_44_0 is also logging. For performance reasons, we will not log to the dataset dataset_8_2 until it is the only active dataset.
2026-05-19 17:58:03,968	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_8_2 =======
2026-05-19 17:58:03,971	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:58:03,973	INFO logging_progress.py:227 -- Active & requested resources: 0/2 CPU, 0.0B/946.0MiB object store
2026-05-19 17:58:03,977	INFO logging_progress.py:192 -- ============================================
2026-05-19 17:58:04,003	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_8_2 execution finished in 0.00 seconds
2026-05-19 17:58:04,027	INFO logging.py:416 -- Registered dataset logger for dataset dataset_31_2
2026-05-19 17:58:04,030	INFO logging.py:424 -- dat

  Station A: Avg Temp: 11.9C, Avg Heat Index: 17.9
  Station B: Avg Temp: 11.3C, Avg Heat Index: 17.3
  Station C: Avg Temp: 11.6C, Avg Heat Index: 17.7
  Station D: Avg Temp: 11.4C, Avg Heat Index: 17.3


---

## 8. Ray Data vs Spark Comparison

In [27]:
comparison = pd.DataFrame({
    "Feature": [
        "Primary use case",
        "Data model",
        "Execution",
        "SQL support",
        "ML integration",
        "Best for"
    ],
    "Spark": [
        "ETL, data warehousing",
        "DataFrame (tabular)",
        "JVM-based",
        "Excellent (Spark SQL)",
        "MLlib (limited)",
        "Joins, aggregations, SQL"
    ],
    "Ray Data": [
        "ML preprocessing",
        "Dataset (row-based)",
        "Native Python",
        "Limited",
        "Excellent (Ray Train)",
        "Feature engineering, batching"
    ]
})

print("Ray Data vs Spark DataFrame")
print("=" * 70)
print(comparison.to_string(index=False))

Ray Data vs Spark DataFrame
         Feature                    Spark                      Ray Data
Primary use case    ETL, data warehousing              ML preprocessing
      Data model      DataFrame (tabular)           Dataset (row-based)
       Execution                JVM-based                 Native Python
     SQL support    Excellent (Spark SQL)                       Limited
  ML integration          MLlib (limited)         Excellent (Ray Train)
        Best for Joins, aggregations, SQL Feature engineering, batching


---

## Summary

### Key Methods

| Method | Purpose | Example |
|--------|---------|--------|
| `map()` | Transform each row | `ds.map(lambda r: {...})` |
| `filter()` | Select rows | `ds.filter(lambda r: r["x"] > 0)` |
| `flat_map()` | One to many | `ds.flat_map(lambda r: [...])` |
| `map_batches()` | Vectorized ops | `ds.map_batches(func)` |
| `groupby()` | Aggregations | `ds.groupby("col").mean()` |

### Best Practices

1. Use `map_batches()` for numeric operations (vectorized)
2. Use `batch_format="pandas"` or `"numpy"` based on your operation
3. For ML: use `train_test_split()` and `iter_batches()`
4. Use Spark for complex joins/SQL; Ray Data for ML preprocessing

### Next: Ray Train

See `03c_ray_train.ipynb` for distributed model training.

In [28]:
# Cleanup
import shutil
shutil.rmtree(temp_dir, ignore_errors=True)
ray.shutdown()
print("Cleanup complete.")

Cleanup complete.
